<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Dependencies

# Installation
! pip install pandas pyarrow -q
! pip install numpy -q
! pip install openai -q
! pip install groq -q

# first-party installations
import itertools
import re
import json
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os

# third-party installations
from google.colab import drive
import pandas as pd
import numpy as np
from openai import OpenAI
from openai import APIError
from groq import Groq

In [ ]:
# @title Paths and definitions

# mount GDrive
drive.mount('/content/drive')

# define dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"

# define simulation/api-call result path
RESULTS_PATH = "/content/drive/MyDrive/Thesis/raw/llm_sim_results.csv"

# define simulation/api-call progress path
LOG_PATH = "/content/drive/MyDrive/Thesis/raw/llm_sim_progress.csv"

# models using the OpenAI-API (as used by Cummins)
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# models using the Groq-API (as used by Cummins)
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"
]

# models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07"
}

# define temperature parameters (as used by Cummins)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# define reasoning parameters (as used by Cummins)
REASONING = Literal["low", "high"]

# Prompt style (CoT) (as used by Li et al. 2025)
CoT = Literal[
    "",
    "explain your thought process step by step"
]

# OpenAI API-key
OPENAI_KEY = "api_key_placeholder"

# GROQ API-key
GROQ_KEY = "api_key_placeholder"

# GROQ URL
GROQ_URL ="https://api.groq.com/openai/v1"

# create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]  # when not a reasoning model
    reasoning_effort: Optional[REASONING]  # when a reasoning model
    chain_of_thought: CoT

# prompt template
PROMPT_TEMPLATE = """

You are given a paper submission for a top-tier Machine Learning conference which you need to write a detailed review.
Please read the paper carefully. Once you have finished reading, please provide the following in your review:
First, write a concise summary of the key points and contributions of the paper inside tags.
Next, think critically about the strengths and weaknesses of the submission.
Inside tags, give a numbered list of at least 4 key reasons why this paper should potentially be accepted to the conference.
For each reason, use sub-bullet points to provide detailed arguments and evidence from the paper to support that reason.
Then, inside tags, give a numbered list of at least 4 key reasons why this paper should potentially be rejected from the conference.
Again, for each reason, use sub-bullet points to provide detailed arguments and evidence from the paper to support that reason.
After weighing the reasons for and against, think of some open questions you have about the work.
List your questions inside tags. Remember, as a reviewer your job is to rigorously evaluate the strengths and weaknesses of the work and to provide critical but constructive feedback to the authors.
Be thorough, specific and detailed in your arguments and feedback.
Highlight both the positives and negatives you see, and justify your points carefully with reference to the content of the paper.

""" # adopted from Xu et al.; What code do I need to change cause of this prompt?

In [ ]:
# @title Load data

# define target column
TARGET_COL = "parsed_text"

# read dataset_paper
dataset_paper = pd.read_parquet(DATASET_DIR + "dataset_paper.parquet")

# select columns 'paper_id', 'abstract' and the target column 'parsed_text'
paper_content = dataset_paper[['paper_id', 'abstract', TARGET_COL]]

# prepare Dict for later applications
targets = paper_content.to_dict('records')


# API

In [ ]:
# @title Defintions for the API call

def _schema_to_tool(chain_of_thought: str) -> Dict[str, Any]:

    return {
        "type": "function",
        "function": {
            "name": "response_format",
            "description": "The function used to return the single, structured text response.",
            "parameters": {
                "type": "object",
                "properties": {
                    "final_response": {
                        "type": "string",
                        "description": (
                            f"Generate the complete, continuous peer-review based on the provided content. {chain_of_thought}"
                        )
                    }
                },
                "required": ["final_response"]
            }
        }
    }


# _normalize_output() is not applicable here as I do not work with integer responses


In [ ]:
# @title API Handling

# function to parse JSON from a string and strip DeepSeek R1 reasoning if present
def _extract_json(text: str, model_name: str = "") -> dict:
    if not text:
        return {}

    # DeepSeek R1: strip 'thinking' section if present
    if "deepseek-r1-distill-llama-70b" in model_name:
        # common pattern: <think>...</think>{...} or reasoning text before JSON
        if "</think>" in text:
            text = text.split("</think>", 1)[-1]
        elif "<think>" in text:
            text = text.split("<think>", 1)[-1]
        # otherwise, try to cut at the first { or [ after any long text
        if "{" in text or "[" in text:
            idx = min(
                [i for i in [text.find("{"), text.find("[")] if i != -1]
            )
            text = text[idx:]

    # strip code fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # find first {...} or [...]
    m = re.search(r"(\{.*\}|$begin:math:display$.*$end:math:display$)", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return {}

# fake clients for testing
class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)

        # placeholders for real clients when simulate=False:
        self._openai = None
        self._groq = None

    # condition on simulate=False
    def _maybe_init_clients(self):
        if self.simulate:
            return
        self._openai = OpenAI(api_key=OPENAI_KEY)
        self._groq = Groq(api_key=GROQ_KEY)

    # define API call
    def call(
        self,
        model: str,
        prompt: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        CoT: Optional[str],
        max_retries: int = 3,
        retry_delay: float = 1.5,
        ) -> Tuple[str, Dict]: # output raw text and the conditions

        self._maybe_init_clients()

        # random values for test sims
        if self.simulate:

          # construct input parameters
          payload = {
              "model": model,
              "temperature": temperature,
              "reasoning_effort": reasoning_effort,
              "CoT_dimension": CoT,
              "simulated_review": (
                  f"This is a simulated review: Model='{model}', Temp={temperature}, "
                  f"Effort='{reasoning_effort}', CoT='{CoT}'."
              )
          }

          # construct output parameter
          raw = json.dumps(payload, indent=2)

          # Immediately return the raw string and the structured dictionary
          return raw, payload

        provider = "groq" if model in GROQ_MODELS else "openai"

        for attempt in range(1, max_retries + 1):
            try:
                if provider == "openai":
                    # If an OpenAI "reasoning" model, use Responses API
                    # and rely on the contract + normalizer.
                    # Otherwise, use Chat Completions with tool-calling to enforce structure strictly.
                    if model in REASONING_MODELS:
                        kwargs = {}
                        if reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # instruction since we can't pass response_format:
                        contract = "Return exactly ONE JSON object" # altered from Jamie as I do not differ input of scales
                        resp = self._openai.responses.create(
                            model=model,
                            input=(prompt + contract),
                            **kwargs,
                        )
                        raw = getattr(resp, "output_text", None) or str(resp)

                    else:
                        # Chat Completions + tool calling (strict) for non-reasoning OpenAI models
                        tools = [_schema_to_tool(CoT)]

                        ckwargs = {}
                        if temperature is not None:
                            ckwargs["temperature"] = float(temperature)

                        # Add nudge to return only a tool call
                        system_msg = {
                            "role": "system",
                            "content": "You are a parser. Call the function with exactly one JSON object that includes a text."
                        }

                        msgs = [system_msg, {"role": "user", "content": prompt}]
                        resp = self._openai.chat.completions.create(
                            model=model,
                            messages=msgs,
                            tools=tools if tools else None,
                            **ckwargs,
                        )

                        choice = resp.choices[0]
                        # Prefer tool call arguments (strict JSON)
                        if choice.message.tool_calls:
                            args = choice.message.tool_calls[0].function.arguments
                            raw = args  # raw JSON text
                        else:
                            # Fallback to message content (should be JSON anyway)
                            raw = choice.message.content or ""

                else:
                    prompt2 = prompt + "Return exactly ONE JSON object" # altered from Jamie as I do not differ input of scales
                    gkwargs = {}
                    if temperature is not None and model not in REASONING_MODELS:
                        gkwargs["temperature"] = float(temperature)
                    resp = self._groq.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt2}],
                        **gkwargs,
                    )
                    raw = resp.choices[0].message.content.strip()

                # Parse + normalize
                parsed_raw = _extract_json(raw, model_name=model)
                return raw, parsed_raw # further downstream this might need the 'normalized_raw' key

            except Exception as e:
                print(f"[LLM ERROR] provider={provider} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    err_stub = f"ERROR: {type(e).__name__}: {e}"
                    return err_stub, {}

            time.sleep(retry_delay)



# Logging handlers

In [ ]:

def _fmt_combo(combo: ParamCombo) -> str:
    return (f"model={combo.model}, temperature={combo.temperature}, "
            f"reasoning_effort={combo.reasoning_effort}, "
            f"prompt_content={combo.CoT}")

def log_call(combo: ParamCombo, **context):
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)


# Define simulation

In [ ]:

# define main simulation workflow
def simulate_parameter_combo(
    combo: ParamCombo,
    review_targets: List[Dict], # altered for my own purpose
    client: Optional[LLMClient] = None,
) -> pd.DataFrame:

    # if client specified then use client, otherwise fake client for test
    client = client or LLMClient(simulate=True)

    records = []
    for review_item in review_targets:
      complete_prompt = f"{PROMPT_TEMPLATE} {combo.CoT}" # altered for my own purpose
      doc_id = review_item["paper_id"]
      log_call(doc_id, combo)

      raw, parsed = client.call(
          model=combo.model,
          prompt=complete.prompt,
          temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
          reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
          chain_of_thought=combo.CoT,
      )

      records.append({
          "document_id": doc_id,
          "review_text": raw,
          "model": combo.model,
          "temperature": combo.temperature,
          "reasoning_effort": combo.reasoning_effort,
          "chain_of_thought": combo.CoT,
      })

    df = pd.DataFrame.from_records(records)
    return df


# Prepare configurations

In [ ]:

# complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS

# generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "CoT"])

# conditional deletion weather a model has a temperature or reasoning paramter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")
display(experiment_config)

print("\nConfiguration table completed!")


# Run simulation

In [ ]:

# Helper to key a combo row (NaN-safe)
def _nan_to_none(x):
    # Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None
    try:
        import pandas as pd  # local import ok in notebooks
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

def combo_key_tuple(row) -> tuple:
    return (
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["CoT"]
    )

def combo_key_str(row) -> str:
    t = combo_key_tuple(row)
    return f"model={t[0]}|temp={t[1]}|re={t[2]}|cot={t[3]}"

# Build the "done" set
done_keys = set()

# read log-path and update done_keys
if os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r") as f:
        for line in f:
            key = line.strip()
            if key:
                done_keys.add(key)

# infer completed combos from RESULTS_PATH, but only if fully complete
if os.path.exists(RESULTS_PATH):
    existing_df = pd.read_csv(RESULTS_PATH)
    if not existing_df.empty:
        EXPECT_ROWS_PER_COMBO = len(experiment_config)*len(paper_content["paper_id"]) # adjusted for my own setting

        # Normalize NA for comparison
        ex = existing_df.copy()
        # Ensure these columns exist (they should)
        needed = ["model","temperature","reasoning_effort","CoT"] #adjusted for my own setting
        missing_cols = [c for c in needed if c not in ex.columns]
        if not missing_cols:
            # normalize NAs to None for keys
            for c in ["temperature","reasoning_effort"]:
                ex[c] = ex[c].where(pd.notna(ex[c]), None)

            # count rows per combo
            counts = (ex.groupby(needed)
                        .size()
                        .reset_index(name="nrows"))

            # Only mark as done if all rows for that combo are completed
            for _, r in counts.iterrows():
                if EXPECT_ROWS_PER_COMBO is not None and int(r["nrows"]) == EXPECT_ROWS_PER_COMBO:
                    done_keys.add(combo_key_str(r))

print(f"Already-completed combos: {len(done_keys)}")

# Iterate with resume capability
client = LLMClient(simulate=False)  # simulate=True sets this to testing mode

new_rows_count = 0
for idx, row in grid_df.iterrows():
    key = combo_key_str(row)

    if key in done_keys:
        print(f"[SKIP] {key} already complete.")
        continue

    combo = ParamCombo(
        model=row["model"],
        temperature=None if pd.isna(row["temperature"]) else float(row["temperature"]),
        reasoning_effort=None if pd.isna(row["reasoning_effort"]) else str(row["reasoning_effort"]),
        chain_of_thought=row["CoT"]
    )

    print(f"\n[RUN] {idx+1}/{len(grid_df)} -> {key}", flush=True)
    try:
        df_combo = simulate_parameter_combo(combo, targets, client=client) # adopted for my own setting

        # Ensure combo columns are present
        df_combo["model"] = combo.model
        df_combo["temperature"] = combo.temperature
        df_combo["reasoning_effort"] = combo.reasoning_effort
        df_combo["CoT"] = combo.chain_of_thought

        # save to disc immediately
        if os.path.exists(RESULTS_PATH): # if exists
            df_combo.to_csv(RESULTS_PATH, mode="a", header=False, index=False) # append
        else: # if not exists
            df_combo.to_csv(RESULTS_PATH, index=False) # write new
        new_rows_count += len(df_combo) # increase count

        # only write after successful completion
        with open(LOG_PATH, "a") as f:
            f.write(key + "\n")

        # Mark as done
        done_keys.add(key)

    except Exception as e:
        print(f"[ERROR] {key} -> {type(e).__name__}: {e}", flush=True)


# Read and process experimental settings

In [ ]:

done_keys = set()

if os.path.exists(RESULTS_PATH):
  existing_df = pd.read_csv(RESULTS_PATH)
  if not existing_df.empty:
    # Normalize NA for comparison
    ex = existing_df.copy()
    ex["temperature"] = ex["temperature"].where(pd.notna(ex["temperature"]), None)
    ex["reasoning_effort"] = ex["reasoning_effort"].where(pd.notna(ex["reasoning_effort"]), None)
    done_keys = set(
        ex.groupby(["model","temperature","reasoning_effort","CoT"])
        .size()
        .reset_index()[["model","temperature","reasoning_effort","CoT"]]
        .apply(lambda r: combo_key_str(r), axis=1)
        .tolist()
        )


# References

